# 고객문의 텍스트 수집 및 전처리

**환경:** Google Colab / Jupyter Notebook (Python 3)  
**핵심:** pandas · re · scikit-learn · kiwipiepy  
**결과물:** `cleaned_customer_inquiries.csv`

### 학습 목표
1. 고객문의 데이터를 불러와 결측치·중복을 점검한다.
2. 문장, 공백 기반 단어, scikit-learn 토큰을 비교한다.
3. 정규표현식으로 개인정보 마스킹·문자 정제를 수행한다.
4. 정규화·불용어 처리와 한국어 형태소 분석의 차이를 이해한다.
5. scikit-learn에 Kiwi 토크나이저를 연결하고 CSV로 저장한다.

> **화장품 상품평 엑셀 파일(3,001건)**을 사용합니다. 실행 전에 엑셀 파일을 업로드하고 `excel_path`를 실제 파일명으로 맞추세요. 형태소 분석기 설치는 최초 1회 인터넷 연결이 필요합니다. 아래 셀을 위에서부터 순서대로 실행하세요.

## 1. 환경 준비 및 데이터 확인

### 실습 흐름
`패키지 준비 → 엑셀 데이터 불러오기 → 데이터 품질 확인 → 분석용 복사본 생성`

**핵심 기능**
- `importlib.util.find_spec()` : 패키지 설치 여부 확인
- `pd.read_excel(usecols=...)` : 필요한 열만 읽어 메모리·시간 절약
- `isna()` / `duplicated()` : 결측값과 중복값 확인
- `dropna()` / `drop_duplicates()` : 분석에서 제외할 행 정리

**확인할 점:** 기존 코드를 그대로 쓰기 위해 `INDEX` → `문의번호`, `상품평` → `원본텍스트`로 컬럼명을 바꿉니다.

In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 15.0 MB/s eta 0:00:00


In [2]:
# 표준 라이브러리 불러오기
import importlib.util  # 모듈 설치 여부 확인
import subprocess      # pip 명령 실행
import sys             # 현재 파이썬 실행 경로 확인

# 실습에 사용할 라이브러리 불러오기
import pandas as pd                                          # 표 형태 데이터 처리
import re                                                    # 정규표현식
from sklearn.feature_extraction.text import CountVectorizer  # 토큰화·단어 빈도
from kiwipiepy import Kiwi                                   # 한국어 형태소 분석기

# 설치 결과 확인 출력
print("pandas:", pd.__version__)
print("실습 환경 준비 완료")

pandas: 2.2.3
실습 환경 준비 완료


- `엑셀파일명.csv`를 Colab 왼쪽 **왼쪽 파일 창(📁)에 끌어다 놓으면** 그 데이터를 씁니다.

In [4]:
# 표 다루기 도구
import pandas as pd

import glob

# ===== ✏️ [입력] 찾을 엑셀 파일 이름 (앞부분만 써도 됨, *는 '아무 글자') =====
excel_path = glob.glob("TS_*.xlsx")[0]     # [0]: 반환된 리스트의 첫 번째 경로를 꺼냄
print(excel_path)

TS_2-4.남성화장품.xlsx


In [5]:

# 필요한 열(INDEX, 상품평)만 문자열로 읽기
df = pd.read_excel(excel_path, usecols=["INDEX", "상품평"], dtype=str)
# 이후 코드와 맞추기 위해 컬럼명 변경
df = df.rename(columns={"INDEX": "문의번호", "상품평": "원본텍스트"})

# 데이터 크기와 앞부분 8행 출력
print("데이터 크기:", df.shape)
display(df.head(8))

데이터 크기: (3001, 2)


,문의번호,원본텍스트
0,217128,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...
1,300582,역시 전에 쓰던 게 좋은 것 같아요.\n향도 독하고 피부에 자극이 강해서 별로예요.
2,300589,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요~~♡♡\n아참...
3,300591,"스킨 2개, 로션 1개 구성으로 3세트\n가성비도 좋고 항상 스킨이 부족하였는데 ..."
4,300598,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다. 제형의 농도가 짙고 착 감...
5,300599,향도 은은하고 남자들이 선호하는 스타일입니다\n명품 화장품을 3세트나 득템했네요^^
6,300610,잘받았읍니다...\n이런 향기였다면 주문 안했을거에요..\n싸구려 향기..\n사우나...
7,300615,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입


In [6]:
# 결측값과 중복 문의 개수 출력
print("결측값 수:", df["원본텍스트"].isna().sum())
print("중복 문의 수:", df["원본텍스트"].duplicated().sum())

# 결측·중복 행을 제외한 작업용 복사본 생성 (원본 df는 유지)
work = df.dropna(subset=["원본텍스트"]).drop_duplicates(subset=["원본텍스트"]).copy()

# 텍스트 컬럼을 문자열 타입으로 변환
work["원본텍스트"] = work["원본텍스트"].astype(str)

# 정리 후 행 수 출력
print("정리 후 데이터 수:", len(work))

결측값 수: 0
중복 문의 수: 0
정리 후 데이터 수: 3001


## 2. 문장·단어 단위 처리

- **문장 분리**: 문장 종결부호 기준의 단순 규칙(완전한 한국어 문장분리기는 아님)
- **공백 단위**: 조사·어미가 어절에 붙어 있음
- **CountVectorizer**: 기본 정규표현식은 글자 2개 이상인 토큰을 추출하므로 한 글자 토큰이 제외될 수 있음

### 실습 흐름
`예제 문장 → 문장 분리 → 공백 분리 → scikit-learn 기본 토큰 확인 → 결과 비교`

**핵심 기능**
- `re.split()` : 지정한 문장 경계 규칙에 따라 문자열 분리
- `str.split()` : 공백을 기준으로 어절 분리
- `CountVectorizer().build_analyzer()` : scikit-learn의 전처리·토큰화 결과 확인

**확인할 점:** `CountVectorizer`는 한국어 조사와 어미를 자동으로 형태소 분석하지 않습니다. 문장 분리 정규식도 교육용 단순 규칙입니다.

In [7]:
# 비교에 사용할 예제 문장
sample = "배송이 늦었습니다. 언제 받을 수 있나요?"


# 공백 기준 어절 분리
words = sample.split()

# CountVectorizer 기본 설정의 토큰화 함수
analyzer = CountVectorizer().build_analyzer()

# 세 방식의 결과 출력
print("문장:", sample)
print("공백 기반:", words)
print("scikit-learn 기본:", analyzer(sample))

문장: 배송이 늦었습니다. 언제 받을 수 있나요?
공백 기반: ['배송이', '늦었습니다.', '언제', '받을', '수', '있나요?']
scikit-learn 기본: ['배송이', '늦었습니다', '언제', '받을', '있나요']


## 3. 정규표현식으로 텍스트 정제

### 실습 흐름
`원본 텍스트 → 이메일·전화번호 치환 → 특수문자 정리 → 연속 공백 정리 → 원본과 비교`

**핵심 기능**
- `re.sub(패턴, 대체문자열, 텍스트)` : 패턴에 맞는 부분 치환
- `apply(함수)` : DataFrame의 각 문장에 사용자 정의 함수 적용
- `display()` / `head()` : 전후 결과를 표로 확인

**확인할 점:** 개인정보에 해당하는 패턴부터 치환합니다. 이 예제의 정규표현식은 대표적인 형식만 처리하며 실제 서비스의 모든 형태를 포괄하지 않습니다.

In [9]:
# 개인정보 마스킹 → 특수문자 제거 → 공백 정리를 수행하는 함수
def clean_text(text):
    # 입력값을 문자열로 변환
    text = str(text)
    # 이메일 주소를 임시 토큰으로 치환
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " EMAILTOKEN ", text)
    # 휴대폰 번호(010 등, 하이픈·공백 선택)를 임시 토큰으로 치환
    text = re.sub(r"(?<!\d)01[016789][- ]?\d{3,4}[- ]?\d{4}(?!\d)", " PHONETOKEN ", text)
    # 한글·영문·숫자·공백 외 문자를 공백으로 치환
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)
    # 연속 공백을 하나로 줄이고 앞뒤 공백 제거
    text = re.sub(r"\s+", " ", text).strip()
    # 임시 토큰을 [EMAIL], [PHONE] 표기로 변환
    return text.replace("EMAILTOKEN", "[EMAIL]").replace("PHONETOKEN", "[PHONE]")

# 모든 행에 정제 함수 적용
work["정제텍스트"] = work["원본텍스트"].apply(clean_text)
# .apply(clean_text) 각 원소를 clean_text에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# 원본과 정제 결과 비교 출력 (앞부분 10행)
display(work[["원본텍스트", "정제텍스트"]].head(10))

,원본텍스트,정제텍스트
0,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...
1,역시 전에 쓰던 게 좋은 것 같아요.\n향도 독하고 피부에 자극이 강해서 별로예요.,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요
2,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요~~♡♡\n아참...,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...
3,"스킨 2개, 로션 1개 구성으로 3세트\n가성비도 좋고 항상 스킨이 부족하였는데 ...",스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...
4,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다. 제형의 농도가 짙고 착 감...,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...
5,향도 은은하고 남자들이 선호하는 스타일입니다\n명품 화장품을 3세트나 득템했네요^^,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요
6,잘받았읍니다...\n이런 향기였다면 주문 안했을거에요..\n싸구려 향기..\n사우나...,잘받았읍니다 이런 향기였다면 주문 안했을거에요 싸구려 향기 사우나에서도 요즘 이런거...
7,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입
8,빠른배송\n여름도 다가오고해서 스킨이 필요했는데\n향기도 좋구 제품구성도 만족합니다...,빠른배송 여름도 다가오고해서 스킨이 필요했는데 향기도 좋구 제품구성도 만족합니다 제...
9,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요^^~~,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요


## 4. 표현 정규화 및 불용어 처리

- 표기 변형을 통일하면 동일한 키워드를 비교하기 쉬워집니다.
- 불용어는 **업무 목적에 따라** 결정합니다. 특히 `안`, `못`, `없다` 같은 부정 표현은 함부로 제거하지 않습니다.

### 실습 흐름
`정제 문장 → 표기 통일 → 불용어 목록 정의 → 처리 전후 토큰 비교`

**핵심 기능**
- `str.replace()` : 특정 표현을 통일된 표현으로 교체
- `CountVectorizer(stop_words=...)` : 지정한 토큰을 분석 과정에서 제외
- `build_analyzer()` : 불용어가 적용된 토큰 목록 확인

**확인할 점:** 불용어 제거는 원본 문장을 직접 삭제하는 것이 아니라 **토큰 분석 단계**에서 적용됩니다. 부정 표현은 중요한 의미를 담을 수 있습니다.

In [10]:
# 지정한 표기 변형을 통일하는 함수
def normalize_text(text):
    # 붙어 있는 복합어를 띄어 쓴 형태로 통일
    text = text.replace("택배배송", "택배 배송")
    text = text.replace("AS문의", "AS 문의")
    # 연속 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 모든 행에 정규화 함수 적용
work["정규화텍스트"] = work["정제텍스트"].apply(normalize_text)
# .apply(normalize_text) 각 원소를 normalize_text에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# 어절 단위 불용어 목록 추가 가능^^
stop_words = ["안녕하세요", "부탁드립니다", "부탁드려요"]

# 불용어를 적용한 CountVectorizer 생성
cv = CountVectorizer(stop_words=stop_words)

# 불용어 적용 전후 토큰 비교
stop_test = "안녕하세요 배송 확인 부탁드립니다"
print("불용어 처리 전:", CountVectorizer().build_analyzer()(stop_test))
print("불용어 처리 후:", cv.build_analyzer()(stop_test))

# 정제 텍스트와 정규화 텍스트 비교 출력
display(work[["정제텍스트", "정규화텍스트"]].head(6))

불용어 처리 전: ['안녕하세요', '배송', '확인', '부탁드립니다']
불용어 처리 후: ['배송', '확인']


,정제텍스트,정규화텍스트
0,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...
1,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요
2,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...
3,스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...,스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...
4,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...
5,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요


## 5. 한국어 형태소 분석과 scikit-learn 연동

Kiwi는 **형태소 분석**을 담당하고 scikit-learn의 `CountVectorizer`는 **분석 결과를 받아 토큰화 인터페이스와 문서-단어 행렬**을 제공합니다. 여기서는 3단원 수치화 내용을 선행하지 않도록 토큰 목록 비교에 초점을 둡니다.

### 실습 흐름
`예제 문장 → Kiwi 형태소·품사 확인 → 필요한 품사 선택 → 사용자 정의 토크나이저 → scikit-learn 연결`

**핵심 기능**
- `Kiwi()` / `kiwi.tokenize()` : 한국어 형태소와 품사 분석
- `token.form` / `token.tag` : 형태소 표면형과 품사 태그
- `CountVectorizer(tokenizer=...)` : 외부 형태소 분석 함수를 토큰화 과정에 사용
- `kiwi.tokenize(문장 리스트)` : 여러 문장을 멀티스레드로 한 번에 분석 (행별 `apply`보다 빠름)

**토큰 선택 규칙 (교육용 예시)**
- 유지 품사: `NNG`(일반 명사), `NNP`(고유 명사), `VV`(동사), `VA`(형용사), `SL`(알파벳 — `AS`, 마스킹 토큰 `EMAIL`·`PHONE` 포함)
- 부정 표현 보존: `안`·`못`(MAG), `않`(VX), `없`(VA)
- 불용어는 Kiwi 분석 결과 형태 기준 `(형태, 품사)` 조합으로 지정: `안녕/NNG`, `부탁/NNG`, `드리/VV`
- Kiwi는 동사·형용사 태그에 불규칙 활용 표시(`-R`, `-I`)를 붙이므로(예: `받/VV-R`) 태그 비교 전에 기본 태그로 변환
- 용언은 어간 뒤에 `다`를 붙여 기본형으로 표기(예: `받` → `받다`)

**확인할 점:** 부정 표현 보존은 문의 내용 해석에 유리하지만, 키워드 빈도 분석에서는 `안`·`못`이 상위 빈도로 나타날 수 있습니다. 품사·불용어 정책은 분석 목적에 따라 조정합니다.

In [11]:
# Kiwi 형태소 분석기 객체 생성
kiwi = Kiwi()

# 비교에 사용할 예제 문장
example = "배송이 늦어서 환불하고 싶습니다."

# 공백 분리 결과
print("공백 분리:", example.split())
# CountVectorizer 기본 토큰화 결과
print("CountVectorizer 기본:", CountVectorizer().build_analyzer()(example))
# Kiwi 형태소(form)와 품사(tag) 결과
print("Kiwi 형태소(형태, 품사):", [(t.form, t.tag) for t in kiwi.tokenize(example)])

공백 분리: ['배송이', '늦어서', '환불하고', '싶습니다.']
CountVectorizer 기본: ['배송이', '늦어서', '환불하고', '싶습니다']
Kiwi 형태소(형태, 품사): [('배송', 'NNG'), ('이', 'JKS'), ('늦', 'VA'), ('어서', 'EC'), ('환불', 'NNG'), ('하', 'XSV'), ('고', 'EC'), ('싶', 'VX'), ('습니다', 'EF'), ('.', 'SF')]


In [13]:
# 유지할 기본 품사 태그 (일반명사, 고유명사, 동사, 형용사, 알파벳)
keep_tags = {"NNG", "NNP", "VV", "VA", "SL"}
# 품사와 관계없이 유지할 부정 표현 (형태, 품사) 조합
negation_set = {("안", "MAG"), ("못", "MAG"), ("않", "VX"), ("없", "VA")}
# 제외할 불용어 (형태, 품사) 조합 - Kiwi 분석 결과 형태 기준
stop_pairs = {("안녕", "NNG"), ("부탁", "NNG"), ("드리", "VV")}

# Kiwi 분석 결과(Token 목록)에서 필요한 형태소만 골라 반환하는 함수
def select_tokens(kiwi_tokens):
    tokens = []
    for token in kiwi_tokens:
        # Kiwi가 반환한 기본 품사 태그
        base_tag = token.tag  #'NNG'
        # 형태와 기본 품사를 묶은 비교용 키
        pair = (token.form, base_tag) #('색상', 'NNG')
        # 불용어 조합이면 건너뜀
        if pair in stop_pairs:
            continue
        # 부정 표현이거나 유지 대상 품사이면 토큰 목록에 추가
        if pair in negation_set or base_tag in keep_tags:
            # 용언(VV·VA·VX)은 어간 뒤에 "다"를 붙여 기본형으로 표기
            if base_tag in {"VV", "VA", "VX"}:
                tokens.append(token.form + "다")
            else:
                tokens.append(token.form)
    return tokens

# 문장 하나를 분석해 선택된 토큰을 반환하는 함수 (CountVectorizer 연결용)
def kiwi_tokenizer(text):
    return select_tokens(kiwi.tokenize(text))

# 사용자 정의 토크나이저를 연결한 CountVectorizer 생성
vectorizer = CountVectorizer(
    tokenizer=kiwi_tokenizer,  # Kiwi 기반 토큰화 함수 사용
    token_pattern=None,        # 기본 토큰 정규식 비활성화
    lowercase=False            # 영문 대소문자 유지
)

# 예제 문장의 토큰 결과 출력
print("Kiwi 맞춤형 토큰:", vectorizer.build_analyzer()(example))

# 전체 문장을 한 번에 분석(멀티스레드)한 뒤 토큰 선택
work["분석토큰"] = [select_tokens(t) for t in kiwi.tokenize(work["정규화텍스트"].tolist())]

# 정규화 텍스트와 분석 토큰 비교 출력
display(work[["정규화텍스트", "분석토큰"]].head(10))

Kiwi 맞춤형 토큰: ['배송', '늦다', '환불']


,정규화텍스트,분석토큰
0,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,"[남편, 사다, 광고, 말, 향, 않다, 올드, 남자, 스킨, 로션, 향, 향]"
1,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요,"[전, 쓰다, 좋다, 같다, 향, 독하다, 피부, 자극, 강하다]"
2,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...,"[남편, 선물, 사다, 향기, 좋다, 지인, 추천, 택배, 상자, 안, 상자, 찌그..."
3,스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...,"[스킨, 로션, 구성, 세트, 가성비, 좋다, 스킨, 부족, 구성, 좋다, 향, 좋..."
4,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...,"[좋아하다, 화장품, 효과, 좋다, 화장품, 형, 농도, 짙다, 감기다, 끈적이다,..."
5,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요,"[향, 남자, 선호, 스타일, 명품, 화장품, 세트, 득템]"
6,잘받았읍니다 이런 향기였다면 주문 안했을거에요 싸구려 향기 사우나에서도 요즘 이런거...,"[향기, 주문, 안, 하다, 싸구려, 향기, 사우나, 요즘, 안, 쓰다, 향, 싸구..."
7,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입,"[가격, 저렴하다, 용량, 타사, 비하다, 많다, 차음, 쓰다]"
8,빠른배송 여름도 다가오고해서 스킨이 필요했는데 향기도 좋구 제품구성도 만족합니다 제...,"[빠르다, 배송, 여름, 다가오다, 스킨, 필요, 향기, 좋다, 제품, 구성, 만족..."
9,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요,"[남자, 화장품, 향, 강하다, 편, 많다, 기능, 향, 좋다]"


## 6. 결과 점검과 파일 저장

결과 파일은 다음 단원(BoW, N-gram, TF-IDF)에서 재사용할 수 있습니다. 토큰 목록은 CSV 저장에 적합하도록 공백으로 연결하고, 원본도 함께 보존합니다.

### 실습 흐름
`토큰 목록 → 공백 구분 문자열 → 컬럼 선택 → 데이터 검증 → CSV 저장`

**핵심 기능**
- `" ".join(tokens)` : 리스트의 토큰을 하나의 문자열로 합침
- `assert` : 결과가 필수 조건을 만족하는지 확인
- `to_csv(index=False, encoding="utf-8-sig")` : 한국어 CSV 저장

**확인할 점:** 원본 텍스트와 정제 텍스트를 함께 남겨야 데이터 가공 과정을 추적할 수 있습니다.

In [14]:
# 토큰 리스트를 공백으로 연결한 문자열 컬럼 생성
work["토큰문자열"] = work["분석토큰"].apply(" ".join)

# 저장할 컬럼만 선택해 복사
output = work[["문의번호", "원본텍스트", "정제텍스트", "정규화텍스트", "토큰문자열"]].copy()

# 결과 검증: 행 존재 여부, 결측 여부
assert len(output) > 0
assert output["정규화텍스트"].notna().all()

# 저장 파일 경로
path = "cleaned_customer_inquiries.csv"

# 행 번호 없이 utf-8-sig 인코딩으로 CSV 저장
output.to_csv(path, index=False, encoding="utf-8-sig")

# 저장 결과와 앞부분 10행 출력
print(f"완료: {path} / {len(output)}건")
display(output.head(10))

완료: cleaned_customer_inquiries.csv / 3001건


,문의번호,원본텍스트,정제텍스트,정규화텍스트,토큰문자열
0,217128,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사다 광고 말 향 않다 올드 남자 스킨 로션 향 향
1,300582,역시 전에 쓰던 게 좋은 것 같아요.\n향도 독하고 피부에 자극이 강해서 별로예요.,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요,역시 전에 쓰던 게 좋은 것 같아요 향도 독하고 피부에 자극이 강해서 별로예요,전 쓰다 좋다 같다 향 독하다 피부 자극 강하다
2,300589,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요~~♡♡\n아참...,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요 아참 택배 상...,남편 선물 사다 향기 좋다 지인 추천 택배 상자 안 상자 찌그러지다 오다 깨지다 않...
3,300591,"스킨 2개, 로션 1개 구성으로 3세트\n가성비도 좋고 항상 스킨이 부족하였는데 ...",스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...,스킨 2개 로션 1개 구성으로 3세트 가성비도 좋고 항상 스킨이 부족하였는데 구성이...,스킨 로션 구성 세트 가성비 좋다 스킨 부족 구성 좋다 향 좋다 부드러
4,300598,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다. 제형의 농도가 짙고 착 감...,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다 제형의 농도가 짙고 착 감기...,좋아하다 화장품 효과 좋다 화장품 형 농도 짙다 감기다 끈적이다 없다 흡수 좋다
5,300599,향도 은은하고 남자들이 선호하는 스타일입니다\n명품 화장품을 3세트나 득템했네요^^,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요,향도 은은하고 남자들이 선호하는 스타일입니다 명품 화장품을 3세트나 득템했네요,향 남자 선호 스타일 명품 화장품 세트 득템
6,300610,잘받았읍니다...\n이런 향기였다면 주문 안했을거에요..\n싸구려 향기..\n사우나...,잘받았읍니다 이런 향기였다면 주문 안했을거에요 싸구려 향기 사우나에서도 요즘 이런거...,잘받았읍니다 이런 향기였다면 주문 안했을거에요 싸구려 향기 사우나에서도 요즘 이런거...,향기 주문 안 하다 싸구려 향기 사우나 요즘 안 쓰다 향 싸구려 쎈 향기 어쩌다 없다 꽝
7,300615,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입,가격 저렴하고 용량이 타사에 비해 많아서 차음써보는것입,가격 저렴하다 용량 타사 비하다 많다 차음 쓰다
8,300694,빠른배송\n여름도 다가오고해서 스킨이 필요했는데\n향기도 좋구 제품구성도 만족합니다...,빠른배송 여름도 다가오고해서 스킨이 필요했는데 향기도 좋구 제품구성도 만족합니다 제...,빠른배송 여름도 다가오고해서 스킨이 필요했는데 향기도 좋구 제품구성도 만족합니다 제...,빠르다 배송 여름 다가오다 스킨 필요 향기 좋다 제품 구성 만족 제품 용량 작다
9,300702,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요^^~~,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요,남자화장품은 대체로 향이 강한편이 많은데 기능성에 향도 은은한것이 아주 좋아요,남자 화장품 향 강하다 편 많다 기능 향 좋다



**참고 공식 문서**  
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)  
- [scikit-learn Text Analytics Tutorial](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)  
- [Kiwi (kiwipiepy)](https://github.com/bab2min/kiwipiepy)